# DeepSeek R1 Distill Qwen 32B — pure inference
T4 ×2 / Q4_K_M. No system prompt, tools, retrieval, or agent. MODE A inline; MODE B authenticated temporary Gradio share. Run cells 1–7 to chat; cell 8 explicitly stops the server. With batch_mode enabled, cell 7 waits for Stop notebook or the session deadline.

## 1. Hardware check

In [ ]:
try:
    import types, sys, json
    runtime = types.ModuleType('deepseek_runtime')
    sys.modules[runtime.__name__] = runtime
    exec('"""Runs only inside the Kaggle notebook. No agent or model tools."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport secrets\nimport base64\n\nos.environ["GRADIO_ANALYTICS_ENABLED"] = "False"\n\nMODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"\nREPO = "bartowski/DeepSeek-R1-Distill-Qwen-32B-GGUF"\nFILENAME = "DeepSeek-R1-Distill-Qwen-32B-Q4_K_M.gguf"\nWORK = Path("/kaggle/working/deepseek-host")\nBASE_URL = "http://127.0.0.1:8080"\nSTOP = threading.Event()\nREPORT = {}\nCONFIG = {}\nPROCESS = None\n\n\ndef run(args, **kwargs):\n    return subprocess.run(list(map(str, args)), check=True, text=True, **kwargs)\n\n\ndef save():\n    WORK.mkdir(parents=True, exist_ok=True)\n    (WORK / "summary.json").write_text(json.dumps(REPORT, indent=2))\n\n\ndef notify(stage, **details):\n    REPORT.update(details, stage=stage, run_id=CONFIG.get("run_id"))\n    save()\n    print("HOST_PROGRESS:" + json.dumps(REPORT), flush=True)\n    if STOP.is_set() and stage != "Stopped":\n        raise RuntimeError("Session stopped from the local interface")\n\n\ndef hardware():\n    run(["nvidia-smi"])\n    devices = subprocess.check_output([\n        "nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader,nounits"\n    ], text=True)\n    rows = [r.split(",") for r in devices.strip().splitlines()]\n    if len(rows) != 2 or any("T4" not in r[1] or int(r[2]) < 14500 for r in rows):\n        raise RuntimeError("Required exactly two NVIDIA T4 GPUs with nominal 16 GB each: " + devices)\n    print(Path("/proc/meminfo").read_text().split("SwapCached")[0])\n    run(["free", "-h"])\n    run(["df", "-h"])\n    WORK.mkdir(parents=True, exist_ok=True)\n    candidates = ["/kaggle/temp", "/tmp", "/var/tmp"]\n    candidates += [line.split()[1] for line in Path("/proc/mounts").read_text().splitlines()\n                   if len(line.split()) > 2 and line.split()[2] in {"ext4", "xfs", "overlay"}]\n    observations = []\n    for candidate in dict.fromkeys(candidates):\n        p = Path(candidate).resolve() / "deepseek-scratch"\n        if p.is_relative_to(Path("/kaggle/working")) or str(p).startswith("/kaggle/input"):\n            continue\n        try:\n            p.mkdir(parents=True, exist_ok=True)\n            probe = p / ".writable"\n            probe.write_text("ok")\n            probe.unlink()\n            free = shutil.disk_usage(p).free\n            existing = p / "models" / FILENAME\n            reusable = existing.exists() and existing.with_suffix(".verified.json").exists()\n            observations.append(f"{p}: {free / 2**30:.1f} GiB free")\n            if free >= 30 * 2**30 or (reusable and free >= 3 * 2**30):\n                CONFIG["scratch"] = str(p)\n                REPORT.update(hardware=devices, scratch=str(p), free_bytes=free)\n                save()\n                print("Selected writable scratch:", observations[-1])\n                return\n        except OSError as exc:\n            observations.append(f"{p}: {exc}")\n    raise RuntimeError("No writable scratch with at least 30 GiB free. No model download started. "\n                       + "; ".join(observations))\n\n\ndef build():\n    run([sys.executable, "-m", "pip", "install", "gradio==6.30.0", "gradio_client==2.7.2",\n         "huggingface_hub>=0.30,<3", "requests>=2.32,<3", "psutil>=5.9", "cryptography>=50.0.2,<51"])\n    bootstrap()\n    notify("Building llama.cpp with CUDA")\n    repo = Path(CONFIG["scratch"]) / "llama.cpp"\n    if not (repo / ".git").exists():\n        run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", repo])\n    else:\n        shutdown_server()\n        run(["git", "-C", repo, "fetch", "--depth", "1", "origin", "master"])\n        run(["git", "-C", repo, "merge", "--ff-only", "FETCH_HEAD"])\n    args = ["cmake", "-S", repo, "-B", repo / "build", "-DGGML_CUDA=ON",\n            "-DCMAKE_CUDA_ARCHITECTURES=75", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF"]\n    # Kaggle mounts the real driver outside the toolkit\'s library search paths.\n    import ctypes\n    ctypes.CDLL("libcuda.so.1")\n    drivers = [Path(line.split()[-1]) for line in Path("/proc/self/maps").read_text().splitlines()\n               if "/libcuda.so" in line and "stubs" not in line]\n    if not drivers:\n        raise RuntimeError("CUDA driver loaded but its filesystem path could not be resolved")\n    driver = drivers[0].resolve()\n    print("Actual CUDA driver library:", driver, flush=True)\n    args += [f"-DCUDA_cuda_driver_LIBRARY={driver}", f"-DCMAKE_LIBRARY_PATH={driver.parent}"]\n    run(args)\n    jobs = max(1, min(4, os.cpu_count() or 2))\n    try:\n        run(["cmake", "--build", repo / "build", "--target", "llama-server", "-j", jobs])\n    except subprocess.CalledProcessError:\n        print("Retrying build with one compiler process to reduce host RAM pressure.")\n        run(["cmake", "--build", repo / "build", "--target", "llama-server", "-j", "1"])\n    CONFIG["binary"] = str(repo / "build/bin/llama-server")\n    devices = subprocess.check_output([CONFIG["binary"], "--list-devices"],\n                                      text=True, stderr=subprocess.STDOUT)\n    print(devices)\n    if not all(re.search(rf"CUDA:?{i}\\b", devices) for i in (0, 1)):\n        raise RuntimeError("llama-server did not enumerate both CUDA:0 and CUDA:1")\n    REPORT["llama_commit"] = subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()\n    save()\n\n\ndef bootstrap():\n    # In managed batch runs credentials are generated in the VM, never embedded in source.\n    if CONFIG.get("public_key"):\n        os.environ.setdefault("GRADIO_USER", "model-user")\n        if not os.environ.get("GRADIO_PASSWORD"):\n            try:\n                from kaggle_secrets import UserSecretsClient\n                os.environ["GRADIO_PASSWORD"] = UserSecretsClient().get_secret("GRADIO_PASSWORD")\n            except Exception:\n                os.environ["GRADIO_PASSWORD"] = secrets.token_urlsafe(24)\n        CONFIG["admin_token"] = secrets.token_urlsafe(32)\n\n\ndef publish_access(url):\n    if not CONFIG.get("public_key"):\n        return\n    from cryptography.hazmat.primitives import hashes, serialization\n    from cryptography.hazmat.primitives.asymmetric import padding\n    public = serialization.load_pem_public_key(CONFIG["public_key"].encode())\n    packet = json.dumps({"url": url, "username": os.environ["GRADIO_USER"],\n                         "password": os.environ["GRADIO_PASSWORD"], "admin_token": CONFIG["admin_token"],\n                         "run_id": CONFIG["run_id"]}).encode()\n    encrypted = public.encrypt(packet, padding.OAEP(mgf=padding.MGF1(hashes.SHA256()),\n                                                   algorithm=hashes.SHA256(), label=None))\n    print("HOST_ACCESS:" + base64.b64encode(encrypted).decode(), flush=True)\n\n\ndef download():\n    from huggingface_hub import HfApi, hf_hub_download\n    notify("Checking exact GGUF file and scratch capacity")\n    folder = Path(CONFIG["scratch"]) / "models"\n    folder.mkdir(exist_ok=True)\n    model = folder / FILENAME\n    marker = model.with_suffix(".verified.json")\n    info = HfApi().model_info(REPO, files_metadata=True)\n    file = next(s for s in info.siblings if s.rfilename == FILENAME)\n    expected_size = file.size\n    expected_hash = getattr(file.lfs, "sha256", None)\n    if not expected_size or not expected_hash:\n        raise RuntimeError("Hugging Face did not return file size and SHA256; refusing unverifiable download")\n    verified = json.loads(marker.read_text()) if marker.exists() else {}\n    valid = (model.exists() and model.stat().st_size == expected_size\n             and verified.get("sha256") == expected_hash\n             and verified.get("mtime_ns") == model.stat().st_mtime_ns)\n    if not valid:\n        free = shutil.disk_usage(folder).free\n        if free < max(30 * 2**30, expected_size + 2 * 2**30):\n            raise RuntimeError(f"Scratch only has {free/2**30:.1f} GiB free; need at least 30 GiB. Download stopped.")\n        notify("Downloading Q4_K_M only", model_bytes=expected_size)\n        model = Path(hf_hub_download(REPO, FILENAME, revision=info.sha, local_dir=folder))\n        if model.stat().st_size != expected_size:\n            raise RuntimeError("Incomplete GGUF download")\n        digest = hashlib.sha256()\n        with model.open("rb") as stream:\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                digest.update(chunk)\n        if digest.hexdigest() != expected_hash:\n            raise RuntimeError("GGUF SHA256 mismatch; refusing to load")\n        marker.write_text(json.dumps({"sha256": expected_hash, "mtime_ns": model.stat().st_mtime_ns}))\n    else:\n        print("Complete, previously SHA256-verified file exists; skipping download.")\n    CONFIG["model"] = str(model)\n    REPORT.update(model=MODEL, quantization="Q4_K_M", model_bytes=expected_size, model_revision=info.sha)\n    save()\n\n\ndef shutdown_server():\n    import psutil\n    binary = CONFIG.get("binary")\n    if not binary:\n        return\n    targets = []\n    for proc in psutil.process_iter(["exe", "cmdline"]):\n        try:\n            if proc.info["exe"] == binary or (proc.info["cmdline"] and proc.info["cmdline"][0] == binary):\n                proc.terminate()\n                targets.append(proc)\n        except (psutil.NoSuchProcess, psutil.AccessDenied):\n            pass\n    _, alive = psutil.wait_procs(targets, timeout=10)\n    for proc in alive:\n        proc.kill()\n    psutil.wait_procs(alive, timeout=10)\n\n\ndef start():\n    import requests\n    global PROCESS\n    shutdown_server()\n    help_text = subprocess.check_output([CONFIG["binary"], "--help"], text=True, stderr=subprocess.STDOUT)\n    flash = "on"\n    attempts = [(8192, "all"), (6144, "all"), (4096, "all")]\n    attempts += [(4096, str(n)) for n in (60, 56, 48, 40, 32, 16, 8, 4, 2)]\n    for ctx, layers in attempts:\n        while True:\n            notify("Loading model", context=ctx, gpu_layers=layers, flash_attention=flash)\n            args = [CONFIG["binary"], "--model", CONFIG["model"], "--host", "127.0.0.1", "--port", "8080",\n                    "--device", "CUDA0,CUDA1", "--n-gpu-layers", layers, "--split-mode", "layer",\n                    "--tensor-split", "1,1", "--ctx-size", str(ctx), "--parallel", "1",\n                    "--flash-attn", flash, "--temp", "0.6", "--top-p", "0.95", "--top-k", "0",\n                    "--min-p", "0", "--repeat-penalty", "1", "--jinja", "--reasoning-format", "deepseek",\n                    "--batch-size", "512", "--ubatch-size", "128"]\n            if "--fit" in help_text:\n                args += ["--fit", "off"]\n            log_path = WORK / "llama-server.log"\n            with log_path.open("w") as log:\n                PROCESS = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)\n            (WORK / "server.pid").write_text(str(PROCESS.pid))\n            deadline = time.monotonic() + 900\n            while PROCESS.poll() is None and time.monotonic() < deadline:\n                if STOP.is_set():\n                    shutdown_server()\n                    raise RuntimeError("Session stopped during model loading")\n                try:\n                    response = requests.get(BASE_URL + "/health", timeout=2)\n                    if response.status_code == 200:\n                        CONFIG["context"] = ctx\n                        notify("Backend ready", backend_url=BASE_URL, server_command=args)\n                        return\n                except requests.RequestException:\n                    pass\n                time.sleep(2)\n            shutdown_server()\n            logs = log_path.read_text(errors="replace")\n            print(logs[-16000:])\n            if flash == "on" and re.search(r"flash.*(?:not supported|unsupported|requires)|(?:not supported|unsupported).*flash", logs, re.I):\n                flash = "off"\n                print("Flash attention unsupported; retrying same memory configuration without it.")\n                continue\n            if re.search(r"out of memory|cudaMalloc.*failed|CUDA error.*memory|failed to allocate.*(?:CUDA|device)", logs, re.I):\n                print("CUDA OOM: advancing context/offload fallback in the prescribed order.")\n                break\n            raise RuntimeError("llama-server failed for a non-OOM reason. See llama-server.log: " + logs[-3000:])\n    raise RuntimeError("Insufficient VRAM after context and GPU offload fallbacks; model was not replaced.")\n\n\ndef vram():\n    raw = subprocess.check_output(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total",\n                                   "--format=csv,noheader,nounits"], text=True)\n    REPORT["gpu_vram"] = raw.strip().splitlines()\n    return raw\n\n\ndef completion(prompt, tokens=512):\n    import requests\n    response = requests.post(BASE_URL + "/v1/chat/completions", json={\n        "model": MODEL, "messages": [{"role": "user", "content": prompt}],\n        "max_tokens": tokens, "temperature": 0.6, "top_p": 0.95, "cache_prompt": False\n    }, timeout=600)\n    response.raise_for_status()\n    return response.json()\n\n\ndef validate():\n    run(["curl", "--fail", "--silent", "--show-error", BASE_URL + "/health"])\n    result = completion("Find the smallest positive integer n for which n leaves remainder 2 when divided by 3, and remainder 3 when divided by 5. Verify the result.", 1024)\n    message = result["choices"][0]["message"]\n    print(json.dumps(result, indent=2, ensure_ascii=False))\n    reasoning = message.get("reasoning_content") or ("<think>" in message.get("content", ""))\n    if not reasoning:\n        raise RuntimeError("Validation returned no reasoning trace; inspect GGUF chat template and server log.")\n    REPORT.update(reasoning_verified=True, validation_usage=result.get("usage"), timings=result.get("timings", {}))\n    print(vram())\n    save()\n\n\ndef benchmark():\n    result = completion("Explain briefly how to check whether 97 is prime, then give the conclusion.", 256)\n    REPORT["timings"] = result.get("timings", {})\n    print(vram())\n    notify("Ready for chat", **summary())\n    print(json.dumps(summary(), indent=2, ensure_ascii=False))\n\n\ndef summary():\n    timing = REPORT.get("timings", {})\n    gpu = REPORT.get("gpu_vram", [])\n    return {"Model": MODEL, "Quantization": "Q4_K_M", "Model size": REPORT.get("model_bytes"),\n            "Context": CONFIG.get("context"), "GPU 0 VRAM": gpu[0] if gpu else None,\n            "GPU 1 VRAM": gpu[1] if len(gpu) > 1 else None,\n            "Prompt processing tok/s": timing.get("prompt_per_second"),\n            "Generation tok/s": timing.get("predicted_per_second"),\n            "Backend URL": BASE_URL, "Gradio URL": REPORT.get("gradio_url")}\n\n\ndef shutdown():\n    shutdown_server()\n    notify("Stopped", stopped=True)\n    demo = CONFIG.get("demo")\n    if demo:\n        demo.close()\n\n\ndef keep_alive():\n    """Batch Kaggle runs must stay inside a cell, otherwise the GPU VM exits."""\n    deadline = time.monotonic() + float(CONFIG.get("session_hours", 3)) * 3600\n    print("Session active. Use Stop notebook in the local interface to release GPUs.", flush=True)\n    try:\n        while not STOP.wait(15):\n            if time.monotonic() >= deadline:\n                break\n            if PROCESS and PROCESS.poll() is not None:\n                raise RuntimeError("llama-server exited unexpectedly")\n            notify("Ready for chat")\n    finally:\n        shutdown()\n', runtime.__dict__)
    runtime.CONFIG.update(json.loads('{"mode": "A", "batch_mode": false}'))
    runtime.hardware()
    runtime.notify('Hardware check passed')
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 2. Build llama.cpp

In [ ]:
try:
    runtime.build()
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 3. Download model

In [ ]:
try:
    runtime.download()
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 4. Start server

In [ ]:
try:
    runtime.STOP.clear()
    runtime.start()
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 5. Server validation

In [ ]:
try:
    runtime.validate()
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 6. Gradio UI

In [ ]:
try:
    ui = types.ModuleType('deepseek_ui')
    exec('"""Shared Gradio interface for local clients and the Kaggle host."""\nfrom __future__ import annotations\nimport asyncio\nimport copy\nimport json\nimport time\nimport gradio as gr\n\n\ndef clean_messages(messages):\n    if not isinstance(messages, list) or not messages or len(messages) > 200:\n        raise ValueError("Conversation must contain 1–200 messages.")\n    result = []\n    total_bytes = 0\n    for msg in messages:\n        if not isinstance(msg, dict) or msg.get("role") not in {"user", "assistant"} or not isinstance(msg.get("content"), str):\n            raise ValueError("Only plain user/assistant messages are accepted.")\n        if "reasoning_content" in msg and not isinstance(msg["reasoning_content"], str):\n            raise ValueError("Reasoning history must be plain text.")\n        total_bytes += len(msg["content"].encode("utf-8")) + len(msg.get("reasoning_content", "").encode("utf-8"))\n        if total_bytes > 1024 * 1024:\n            raise ValueError("Conversation exceeds the 1 MiB text limit")\n        result.append({k: msg[k] for k in ("role", "content", "reasoning_content") if k in msg})\n    return result\n\n\ndef display(messages):\n    result = []\n    for msg in messages:\n        content = msg["content"]\n        if msg.get("reasoning_content"):\n            content = "<details><summary>Reasoning</summary>\\n\\n" + msg["reasoning_content"] + "\\n\\n</details>\\n\\n" + content\n        result.append({"role": msg["role"], "content": content})\n    return result\n\n\ndef metrics_text(data):\n    usage, timing = data.get("usage") or {}, data.get("timings") or {}\n    rate = timing.get("predicted_per_second")\n    elapsed = data.get("elapsed", 0)\n    parts = [f"Total: {elapsed:.1f} s"]\n    if timing.get("predicted_ms") is not None:\n        parts.append(f"Generation time: {timing[\'predicted_ms\']/1000:.1f} s")\n    if usage:\n        parts += [f"Input: {usage.get(\'prompt_tokens\', \'?\')} tokens", f"Output: {usage.get(\'completion_tokens\', \'?\')} tokens"]\n    if rate is not None:\n        parts.append(f"Generation: {rate:.2f} tok/s")\n    if timing.get("prompt_per_second") is not None:\n        parts.append(f"Prompt: {timing[\'prompt_per_second\']:.2f} tok/s")\n    if data.get("note"):\n        parts.append(data["note"])\n    return " · ".join(parts)\n\n\nasync def prepare_request(client, base_url, messages, context, max_tokens, behavior):\n    messages = clean_messages(messages)\n    original_length = len(messages)\n    requested = max(1, min(int(max_tokens), 8192))\n    while True:\n        rendered = await client.post(base_url + "/apply-template", json={"messages": messages})\n        rendered.raise_for_status()\n        tokenized = await client.post(base_url + "/tokenize", json={"content": rendered.json()["prompt"], "add_special": False})\n        tokenized.raise_for_status()\n        prompt_tokens = len(tokenized.json()["tokens"])\n        available = context - prompt_tokens - 8\n        reserve = min(requested, context // 2)\n        if behavior == "Keep recent turns" and available < reserve and len(messages) > 2:\n            messages = messages[2:]\n            continue\n        if available <= 0:\n            raise ValueError(f"Prompt exceeds context {context}. Shorten it or start a new chat.")\n        limit = min(requested, available)\n        note = f"Output budget: {limit}/{requested}; context {context}"\n        if len(messages) < original_length:\n            note += f"; omitted {original_length-len(messages)} oldest messages for this request"\n        return messages, limit, note\n\n\ndef make_llama_stream(base_url, context_getter):\n    async def stream(messages, temperature, top_p, max_tokens, behavior):\n        import httpx\n        start = time.monotonic()\n        messages = clean_messages(messages)\n        if messages[-1]["role"] != "user":\n            raise ValueError("Last message must be a user prompt.")\n        async with httpx.AsyncClient(timeout=httpx.Timeout(600, connect=15)) as client:\n            sent, limit, note = await prepare_request(client, base_url, messages, context_getter(), max_tokens, behavior)\n            payload = {"model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B", "messages": sent,\n                       "temperature": max(0, min(float(temperature), 2)), "top_p": max(0.01, min(float(top_p), 1)),\n                       "max_tokens": limit, "stream": True, "stream_options": {"include_usage": True}}\n            result = {"messages": copy.deepcopy(messages) + [{"role": "assistant", "content": "", "reasoning_content": ""}],\n                      "elapsed": 0, "usage": {}, "timings": {}, "note": note}\n            async with client.stream("POST", base_url + "/v1/chat/completions", json=payload) as response:\n                if response.is_error:\n                    raise ValueError((await response.aread()).decode()[:1500])\n                async for line in response.aiter_lines():\n                    if not line.startswith("data: "):\n                        continue\n                    if line[6:].strip() == "[DONE]":\n                        break\n                    data = json.loads(line[6:])\n                    if data.get("error"):\n                        raise ValueError(str(data["error"]))\n                    choices = data.get("choices") or []\n                    delta = choices[0].get("delta", {}) if choices else {}\n                    for key in ("content", "reasoning_content"):\n                        result["messages"][-1][key] += delta.get(key) or ""\n                    result["usage"] = data.get("usage") or result["usage"]\n                    result["timings"] = data.get("timings") or result["timings"]\n                    result["elapsed"] = time.monotonic() - start\n                    yield copy.deepcopy(result)\n            result["elapsed"] = time.monotonic() - start\n            yield result\n    return stream\n\n\ndef build_chat(stream, title="DeepSeek R1 · 32B", controls=None):\n    with gr.Blocks(title=title, analytics_enabled=False) as demo:\n        gr.Markdown(f"# {title}\\nPure LLM inference · Q4_K_M · NVIDIA T4 ×2")\n        history = gr.State([])\n        with gr.Row():\n            with gr.Column(scale=1, min_width=245):\n                if controls:\n                    controls()\n                gr.Markdown("### Settings")\n                temperature = gr.Slider(0, 2, value=0.6, step=0.05, label="Temperature")\n                top_p = gr.Slider(0.05, 1, value=0.95, step=0.01, label="Top p")\n                max_tokens = gr.Slider(1, 8192, value=8192, step=1, label="Max output tokens")\n                behavior = gr.Dropdown(["Keep recent turns", "Keep all / reject overflow"], value="Keep recent turns", label="Context behavior")\n                gr.Markdown("Input and output share the context window. The output budget is capped to remaining space.")\n                new = gr.Button("New Chat")\n                clear = gr.Button("Clear conversation")\n            with gr.Column(scale=4):\n                chat = gr.Chatbot(height="max(260px, calc(100vh - 360px))", label="Conversation", render_markdown=True, sanitize_html=True,\n                                  latex_delimiters=[{"left": "$$", "right": "$$", "display": True},\n                                                    {"left": "\\\\[", "right": "\\\\]", "display": True},\n                                                    {"left": "\\\\(", "right": "\\\\)", "display": False},\n                                                    {"left": "$", "right": "$", "display": False}])\n                prompt = gr.Textbox(label="Message", placeholder="Nhập tin nhắn…", lines=3)\n                with gr.Row():\n                    send = gr.Button("Send", variant="primary")\n                    stop = gr.Button("Stop Generation", variant="stop")\n                    regen = gr.Button("Regenerate")\n                metrics = gr.Markdown("Ready when the Kaggle backend is connected.")\n\n        async def respond(text, previous, temp, p, maximum, context_behavior, regenerate=False):\n            messages = copy.deepcopy(previous or [])\n            if regenerate:\n                if messages and messages[-1]["role"] == "assistant":\n                    messages.pop()\n                if not messages or messages[-1]["role"] != "user":\n                    yield display(messages), messages, "", "No response to regenerate."\n                    return\n            else:\n                if not text.strip():\n                    yield display(messages), messages, text, "Enter a message."\n                    return\n                messages.append({"role": "user", "content": text})\n            yield display(messages), messages, "", "Waiting for the inference slot…"\n            try:\n                async for result in stream(messages, temp, p, maximum, context_behavior):\n                    yield display(result["messages"]), result["messages"], "", metrics_text(result)\n            except asyncio.CancelledError:\n                raise\n            except Exception as exc:\n                yield display(messages), messages, "", f"Error: {str(exc)[:1800]}"\n\n        async def regenerate(previous, temp, p, maximum, context_behavior):\n            async for value in respond("", previous, temp, p, maximum, context_behavior, True):\n                yield value\n\n        inputs = [prompt, history, temperature, top_p, max_tokens, behavior]\n        outputs = [chat, history, prompt, metrics]\n        events = [send.click(respond, inputs, outputs, api_name=False, concurrency_id="inference", concurrency_limit=1),\n                  prompt.submit(respond, inputs, outputs, api_name=False, concurrency_id="inference", concurrency_limit=1),\n                  regen.click(regenerate, inputs[1:], outputs, api_name=False, concurrency_id="inference", concurrency_limit=1)]\n        stop.click(fn=lambda: "Generation stopped.", outputs=metrics, cancels=events, queue=False, api_name=False)\n        for button in (new, clear):\n            button.click(lambda: ([], [], "", "New conversation."), outputs=outputs, cancels=events, queue=False, api_name=False)\n    demo.queue(max_size=32, default_concurrency_limit=1)\n    return demo\n\n\ndef add_host_api(demo, stream, status_fn, stop_fn):\n    with demo:\n        messages = gr.JSON(visible=False)\n        temp = gr.Number(0.6, visible=False)\n        top_p = gr.Number(0.95, visible=False)\n        maximum = gr.Number(8192, visible=False)\n        behavior = gr.Textbox("Keep recent turns", visible=False)\n        result = gr.JSON(visible=False)\n        gr.Button(visible=False).click(stream, [messages, temp, top_p, maximum, behavior], result,\n                                       api_name="infer", concurrency_id="inference", concurrency_limit=1)\n        gr.Button(visible=False).click(status_fn, outputs=result, api_name="host_status", queue=False)\n        admin = gr.Textbox(visible=False)\n        gr.Button(visible=False).click(stop_fn, admin, result, api_name="shutdown", queue=False)\n\n\ndef launch_host(runtime):\n    import os\n    stream = make_llama_stream(runtime.BASE_URL, lambda: runtime.CONFIG["context"])\n    demo = build_chat(stream)\n    mode = os.environ.get("GRADIO_MODE", runtime.CONFIG.get("mode", "A")).upper()\n    if mode not in {"A", "B"}:\n        raise ValueError("GRADIO_MODE must be A or B")\n    # Hosted Kaggle notebooks cannot embed a VM\'s localhost URL in the user\'s browser.\n    # Gradio\'s documented hosted-notebook transport uses a share tunnel for the iframe.\n    share = mode == "B" or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))\n    username = os.environ.get("GRADIO_USER", "model-user")\n    password = os.environ.get("GRADIO_PASSWORD")\n    if share and not password:\n        try:\n            from kaggle_secrets import UserSecretsClient\n            password = UserSecretsClient().get_secret("GRADIO_PASSWORD")\n        except Exception:\n            pass\n    if mode == "B" and not password:\n        raise RuntimeError("MODE B requires GRADIO_PASSWORD environment variable or Kaggle Secret.")\n    if share and not password:\n        import secrets\n        password = secrets.token_urlsafe(24)\n        os.environ["GRADIO_PASSWORD"] = password\n        print("Inline login password generated in GRADIO_PASSWORD. Retrieve it privately from the runtime environment; do not publish it in notebook output.")\n\n    def host_status():\n        runtime.vram()\n        return dict(runtime.REPORT, **runtime.summary())\n\n    def stop_host(token):\n        import secrets\n        if not runtime.CONFIG.get("admin_token") or not secrets.compare_digest(str(token), runtime.CONFIG["admin_token"]):\n            raise ValueError("Invalid admin token")\n        runtime.STOP.set()\n        return {"stopping": True}\n\n    add_host_api(demo, stream, host_status, stop_host)\n    old = runtime.CONFIG.get("demo")\n    if old:\n        old.close()\n    _, local_url, share_url = demo.launch(server_name="127.0.0.1", server_port=7860, share=share,\n                                          auth=(username, password) if share else None,\n                                          inline=mode == "A", prevent_thread_lock=True, quiet=True,\n                                          show_error=False, mcp_server=False, strict_cors=True,\n                                          blocked_paths=["/kaggle", "/root", "/tmp", "/var/tmp"])\n    runtime.CONFIG["demo"] = demo\n    if share and not share_url:\n        raise RuntimeError("Gradio share tunnel failed; localhost remains private. Retry the UI cell.")\n    runtime.notify("Gradio connected", gradio_url=share_url or local_url)\n    runtime.publish_access(share_url or local_url)\n    return demo\n', ui.__dict__)
    demo = ui.launch_host(runtime)
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 7. Benchmark

In [ ]:
try:
    runtime.benchmark()
    # Keep batch execution alive; Stop notebook releases the GPU VM.
    if runtime.CONFIG.get('batch_mode', False):
        runtime.keep_alive()
except Exception as exc:
    import traceback
    traceback.print_exc()
    runtime.REPORT.update(error=str(exc), stage='Failed', run_id=runtime.CONFIG.get('run_id'))
    runtime.save()
    print('HOST_PROGRESS:' + json.dumps(runtime.REPORT), flush=True)
    if runtime.CONFIG.get('binary'):
        runtime.shutdown_server()
    raise


## 8. Shutdown / cleanup

In [ ]:
# Safe to run manually. Scratch model is kept for reruns within this VM.
runtime.STOP.set()
runtime.shutdown()
print(json.dumps(runtime.summary(), indent=2))